In [0]:

# Secure Widgets (DO NOT COMMIT SECRETS)


dbutils.widgets.text("eh_conn_string", "", "Event Hub Connection String")
dbutils.widgets.text("storage_key", "", "Storage Account Key")

event_hub_conn_str = dbutils.widgets.get("eh_conn_string")
storage_key = dbutils.widgets.get("storage_key")

# Safety check
if not event_hub_conn_str or not storage_key:
    raise ValueError("Please enter secrets in the widgets at the top of the notebook!")


In [0]:
import json
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
#Schema of dataset

schema = StructType([


    StructField('order_id',StringType()),
    StructField('timestamp',StringType()),
    StructField('customer_id',StringType()),
    StructField('product_id',StringType()),
    StructField('category',StringType()),
    StructField('price',DoubleType()),
    StructField('quantity',IntegerType()),
    StructField('total_amount',DoubleType()),
    StructField('city',StringType()),
    StructField('state',StringType()),
    StructField('country',StringType()),
    StructField('latitude',StringType()),
    StructField('longitude',StringType()),
    StructField('delivery_status',StringType()),
])

In [0]:
# Azure Event Hub Configurations
event_hub_namespace = "ecommerce-namespace-77.servicebus.windows.net"
event_hub_name = "ecommerce-order"


In [0]:
eh_conf = {
    'kafka.bootstrap.servers': f"{event_hub_namespace}:9093",
    'subscribe': event_hub_name,
    'kafka.security.protocol': 'SASL_SSL',
    'kafka.sasl.mechanism': 'PLAIN',
    'kafka.sasl.jaas.config': f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{event_hub_conn_str}";',
    'startingOffsets': 'latest',
    'failOnDataLoss': 'false'
}

In [0]:
df_raw=spark.readStream.format("kafka").options(**eh_conf).load()

## parse json from  kafka streams
df_orders=(
    df_raw.selectExpr("cast(value as string) as jsonData")
    .select(from_json("jsonData",schema).alias("data"))
    .select('data.*')
)


In [0]:


spark.conf.set(
  "fs.azure.account.key.ecommercestorage77.dfs.core.windows.net",
  storage_key
)

In [0]:
bronze_path = "abfss://e-commerce@ecommercestorage77.dfs.core.windows.net/bronze"

In [0]:
(
    df_orders.writeStream
    .format('delta')
    .outputMode("append")
    .option("checkpointLocation",bronze_path+"/_checkpoint")
    .start(bronze_path)
)

In [0]:
df=spark.read.format("delta").load(bronze_path)
display(df)